<a href="https://colab.research.google.com/github/kharlamsergey/colab/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0%D1%8F_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

netflix = pd.read_csv('/content/netflix_titles.csv')

netflix.drop(columns='gross', inplace=True)
netflix.drop(columns='star', inplace=True)
netflix.drop(columns='writer', inplace=True)
netflix.drop(columns='company', inplace=True)
netflix.drop(columns='director', inplace=True)
netflix.drop(columns='released', inplace=True)
netflix.drop(columns='year', inplace=True)
# Убираем пустые

# Числовые колонки
numeric_cols = ['budget', 'runtime', 'votes', 'score']
for col in numeric_cols:
    if netflix[col].isnull().sum() > 0:
        mean_value = netflix[col].mean()
        netflix[col] = netflix[col].fillna(mean_value)


# Работаем с категориями
rating = pd.get_dummies(netflix['rating'], dtype=int)
genre = pd.get_dummies(netflix['genre'], dtype=int)
country = pd.get_dummies(netflix['country'], dtype=int)

netflix.drop(["rating", "genre", "country", "name"], axis = 1, inplace = True)

netflix = pd.concat([netflix, rating, genre, country], axis = 1)

# Нормализация данных
scaler = StandardScaler()

cols_to_scale = ["votes", "budget", "runtime"]

scaler.fit(netflix[cols_to_scale])
netflix[cols_to_scale] = scaler.transform(netflix[cols_to_scale])

netflix.columns = netflix.columns.map(str)


# Создаём новую целевую переменную — классы
def score_to_class(score):
    if score < 6.0:
        return 0  # Низкий
    elif score < 8.0:
        return 1  # Средний
    else:
        return 2  # Высокий


# Разделение признаков
x_net = netflix.drop("score", axis = 1)
y_net = netflix["score"].apply(score_to_class)

x_net.isnull().sum()

# Обучение модели регрессии
model = LogisticRegression()

model.fit(x_net, y_net)

y_pred_net = model.predict(x_net)

#  Матрица ошибок
conf_matrix = confusion_matrix(y_net, y_pred_net)
conf_matrix_df = pd.DataFrame(conf_matrix)
conf_matrix_df

conf_matrix_labels = pd.DataFrame(conf_matrix, columns = ["Прогноз плохой рейтинг",
"Прогноз хороший рейтинг", "Прогноз отличный рейтинг"], index = ["Факт плохой рейтинг",
"Факт хороший рейтинг", "Факт отличный рейтинг"])

# Проверим точность предстказаний
model_accuracy = accuracy_score(y_net, y_pred_net)
round(model_accuracy, 3)

# Загрузим в файл данные
conf_matrix_labels.to_csv("result.csv", index = True)
files.download('/content/result.csv')


KeyError: "['gross'] not found in axis"